In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-pd-select-best-model'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
boto3==1.24.59

pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '02_pricing_pd'
    
    # load output from tuning
    print('Loading output from tuning...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df = pd.read_csv(str_uri)
    # get iteration
    int_best_iteration = df['iteration'].iloc[0]
    print(f'Best model was from iteration {int_best_iteration}')
    
    # copy model
    print('Copying model...')
    # init
    cls_client = boto3.client('s3')
    # get args
    str_filename = f'dict_model_inference_{int_best_iteration}.pkl'
    str_key_source = f'{str_model}/02_model/02_model/02_batch_tuning/models/{str_filename}'
    dict_copy_source = {
        'Bucket': str_project,
        'Key': str_key_source,
    }
    str_filename_destination = 'final_model.pkl'
    str_key_destination = f'{str_model}/02_model/03_final_model/{str_filename_destination}'
    cls_client.copy_object(
        CopySource=dict_copy_source,
        Bucket=str_project,
        Key=str_key_destination,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-select-best-model

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  57.86kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 303d20c679a3
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d25abc48fcf6
Step 3/6 : COPY requirements.txt  .
 ---> 4b5b3f455e88
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Running in 3a009aec7cef
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 64.5 MB/s eta 0:00:00


Removing intermediate container 3a009aec7cef
 ---> 2e82a0504ad2
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 7b0da196da0d
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in a2ae5613adac
Removing intermediate container a2ae5613adac
 ---> 6e0663c70977
Successfully built 6e0663c70977
Successfully tagged genxii-pd-select-best-model:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-select-best-model' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-select-best-model]
e5ffcfe5f095: Preparing
5dc96ca07a04: Preparing
37b646d4073b: Preparing
d630e2305053: Preparing
3bd433acfe84: Preparing
09b55d38856d: Preparing
e073f5919ae5: Preparing
b3b414f01759: Preparing
8308f08f35ba: Preparing
c8203e562a8c: Preparing
09b55d38856d: Waiting
e073f5919ae5: Waiting
b3b414f01759: Waiting
8308f08f35ba: Waiting
c8203e562a8c: Waiting
e5ffcfe5f095: Pushed
37b646d4073b: Pushed
e073f5919ae5: Pushed
d630e2305053: Pushed
b3b414f01759: Pushed
8308f08f35ba: Pushed
3bd433acfe84: Pushed
09b55d38856d: Pushed
5dc96ca07a04: Pushed
c8203e562a8c: Pushed
latest: digest: sha256:2c0f905882bf341a5a4f706b12fe93ed115f3cf9bb090288329f7e75e3913538 size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:13:59 GMT',
                                      'x-amzn-requestid': 'f519c77a-8f7b-437c-abd5-f0b8c5747256'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'f519c77a-8f7b-437c-abd5-f0b8c5747256',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '2c0f905882bf341a5a4f706b12fe93ed115f3cf9bb090288329f7e75e3913538',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-select-best-model',
 'FunctionName': 'genxii-pd-select-best-model',
 'LastModified': '2024-08-20T16:13:59.138+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-pd-select-best-model'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1213',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:13:59 GMT',
                                      'x-amzn-requestid': 'ea1c8b37-2fc3-4332-88a5-aa94d9e33b42'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'e

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)